# WAHIS Basic EDA

Self-contained first-pass EDA for HPAI records in `data/structured/wahis/Event-list-20_5_2026.csv`. This notebook follows the same structure as the ADIS and EMPRES-i EDA notebooks while keeping WAHIS-specific fields such as event reasons, report status, and report identifiers.

## Setup

Shared plotting, display, and helper functions used throughout the notebook.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt

plt.style.use("ggplot")

params = {
    "text.color": (0.25, 0.25, 0.25),
    "figure.figsize": [16, 9],
}

plt.rcParams.update(params)

# Get colors from default theme.
DEFAULT_COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

import seaborn as sns
# sns.set()


from pathlib import Path

import pandas as pd


pd.options.display.max_columns = 120
pd.options.mode.chained_assignment = None
pd.options.display.max_rows = 500
pd.options.display.max_seq_items = 500

NA_VALUES = ['NaN', 'nan', '', ';']

def find_data_path(*candidates: str) -> Path:
    paths = [Path(candidate) for candidate in candidates]
    path = next((candidate for candidate in paths if candidate.exists()), None)
    if path is None:
        tried = '\n'.join(str(candidate) for candidate in paths)
        raise FileNotFoundError(f'Could not find source CSV. Tried:\n{tried}')
    return path


def schema_summary(data: pd.DataFrame) -> pd.DataFrame:
    return (
        pd.DataFrame({
            'dtype': data.dtypes.astype(str),
            'missing': data.isna().sum(),
            'missing_pct': data.isna().mean().mul(100).round(1),
            'n_unique': data.nunique(dropna=True),
        })
        .sort_values(['missing_pct', 'n_unique'], ascending=[False, False])
    )


def top_counts(data: pd.DataFrame, column: str, n: int = 15) -> pd.DataFrame:
    return (
        data[column]
        .fillna('Missing')
        .value_counts(dropna=False)
        .head(n)
        .rename_axis(column)
        .reset_index(name='records')
    )


def plot_top_counts(data: pd.DataFrame, specs: list[tuple[str, int, str]], cols: int = 2) -> None:
    rows = -(-len(specs) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 4.8 * rows), squeeze=False)
    for axis, (column, n, title) in zip(axes.ravel(), specs):
        counts = top_counts(data, column, n=n)
        sns.barplot(data=counts, y=column, x='records', ax=axis, color='#4C78A8')
        axis.set_title(title)
        axis.set_xlabel('Records')
        axis.set_ylabel('')
    for axis in axes.ravel()[len(specs):]:
        axis.set_visible(False)
    plt.tight_layout()


def date_coverage(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame({
        'date_column': columns,
        'min_date': [data[column].min() for column in columns],
        'max_date': [data[column].max() for column in columns],
        'missing': [data[column].isna().sum() for column in columns],
        'missing_pct': [round(data[column].isna().mean() * 100, 1) for column in columns],
    })


def monthly_counts(data: pd.DataFrame, date_column: str, label: str = 'records') -> pd.DataFrame:
    return (
        data.dropna(subset=[date_column])
        .assign(month=lambda frame: frame[date_column].dt.to_period('M').dt.to_timestamp())
        .groupby('month')
        .size()
        .rename(label)
        .reset_index()
    )


def monthly_counts_by_category(
    data: pd.DataFrame,
    date_column: str,
    category_column: str,
    top_n: int = 6,
    label: str = 'records',
) -> pd.DataFrame:
    top_values = top_counts(data, category_column, n=top_n)[category_column]
    return (
        data[data[category_column].isin(top_values)]
        .dropna(subset=[date_column])
        .assign(month=lambda frame: frame[date_column].dt.to_period('M').dt.to_timestamp())
        .groupby(['month', category_column])
        .size()
        .rename(label)
        .reset_index()
    )


def country_disease_matrix(
    data: pd.DataFrame,
    country_column: str,
    disease_column: str,
    country_n: int = 12,
    disease_n: int = 8,
) -> pd.DataFrame:
    top_countries = top_counts(data, country_column, n=country_n)[country_column]
    top_diseases = top_counts(data, disease_column, n=disease_n)[disease_column]
    filtered = data[data[country_column].isin(top_countries) & data[disease_column].isin(top_diseases)]
    return pd.crosstab(filtered[country_column], filtered[disease_column])

## Load Data

WAHIS uses semicolon-delimited CSV exports and `dd.mm.yyyy`-style dates.

In [ ]:
DATA_PATH = find_data_path(
    '../data/structured/wahis/Event-list-20_5_2026.csv',
    'data/structured/wahis/Event-list-20_5_2026.csv',
)

raw_df = pd.read_csv(DATA_PATH, sep=';', na_values=NA_VALUES, keep_default_na=True)

print(f'Loaded {len(raw_df):,} rows and {raw_df.shape[1]:,} columns from {DATA_PATH}')
raw_df.head()

## Clean Working Copy

Parse dates, filter to HPAI records, add a submission delay, and keep the original source column names so they remain traceable to the export.

In [ ]:
df = raw_df.copy()
df.columns = df.columns.str.strip()

HPAI_MATCH_COLUMNS = ['disease', 'subType']
hpai_mask = pd.Series(False, index=df.index)
for column in HPAI_MATCH_COLUMNS:
    hpai_mask |= df[column].fillna('').str.contains(
        'HPAI|high pathogenic|high pathogenicity avian influenza',
        case=False,
        regex=True,
    )

df = df.loc[hpai_mask].copy()
print(f'Filtered to {len(df):,} HPAI records from {len(raw_df):,} total WAHIS records.')

DATE_COLUMNS = ['eventStartDate', 'submissionDate']
for column in DATE_COLUMNS:
    df[column] = pd.to_datetime(df[column], dayfirst=True, errors='coerce')

df['submission_delay_days'] = (df['submissionDate'] - df['eventStartDate']).dt.days

id_columns = ['eventId', 'reportId', 'reportNumber']
duplicate_summary = pd.Series({
    f'duplicate_{column}': df[column].duplicated().sum()
    for column in id_columns
    if column in df.columns
})

display(df.dtypes.to_frame('dtype'))
display(duplicate_summary.to_frame('count'))

## Dataset Overview And Completeness

In [ ]:
overview = pd.DataFrame({
    'rows': [len(df)],
    'columns': [df.shape[1]],
    'countries': [df['country'].nunique()],
    'diseases': [df['disease'].nunique()],
    'subtypes': [df['subType'].nunique()],
    'event_reasons': [df['reason'].nunique()],
    'first_event_start': [df['eventStartDate'].min()],
    'last_event_start': [df['eventStartDate'].max()],
})

display(overview)
display(schema_summary(df))
display(date_coverage(df, DATE_COLUMNS))

## Top Categories

In [ ]:
CATEGORY_COLUMNS = ['country', 'disease', 'subType', 'reason', 'reportStatus']

for column in CATEGORY_COLUMNS:
    display(top_counts(df, column, n=20))

In [ ]:
plot_top_counts(
    df,
    [
        ('country', 15, 'Top Countries'),
        ('disease', 12, 'Top Diseases'),
        ('reason', 12, 'Event Reasons'),
        ('subType', 12, 'Top Subtypes'),
    ],
)

## Time Coverage And Trends

In [ ]:
events_by_month = monthly_counts(df, 'eventStartDate')

sns.lineplot(data=events_by_month, x='month', y='records', marker='o')
plt.title('WAHIS Records By Event Start Month')
plt.xlabel('Event start month')
plt.ylabel('Records')
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
events_by_month_and_disease = monthly_counts_by_category(
    df,
    date_column='eventStartDate',
    category_column='disease',
    top_n=6,
)

sns.lineplot(
    data=events_by_month_and_disease,
    x='month',
    y='records',
    hue='disease',
    marker='o',
)
plt.title('Top WAHIS Diseases Over Time')
plt.xlabel('Event start month')
plt.ylabel('Records')
plt.xticks(rotation=45)
plt.legend(title='Disease', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

## Delay Analysis

In [ ]:
delay_columns = ['submission_delay_days']

display(df[delay_columns].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).T)

valid_delays = df.loc[df['submission_delay_days'].between(-30, 365), 'submission_delay_days']
sns.histplot(valid_delays, bins=50, color='#59A14F')
plt.title('Submission Delay Distribution')
plt.xlabel('Days from event start to submission')
plt.ylabel('Records')
plt.tight_layout()

In [ ]:
df.loc[
    df['submission_delay_days'].notna(),
    ['country', 'disease', 'subType', 'eventStartDate', 'submissionDate', 'submission_delay_days', 'reason'],
].sort_values('submission_delay_days', ascending=False).head(20)

## Country-Disease Matrix

In [ ]:
matrix = country_disease_matrix(df, 'country', 'disease')

sns.heatmap(matrix, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Top WAHIS Countries And Diseases')
plt.xlabel('Disease')
plt.ylabel('Country')
plt.tight_layout()

## Source-Specific Signals

WAHIS has no coordinates in this export, so the source-specific checks focus on event reasons, report status, identifiers, and latest submissions.

In [ ]:
source_specific = {
    'reason_by_disease': pd.crosstab(df['disease'], df['reason'], dropna=False),
    'status_by_reason': pd.crosstab(df['reason'], df['reportStatus'], dropna=False),
    'reports_per_event': df.groupby('eventId')['reportId'].nunique().sort_values(ascending=False),
}

display(source_specific['reason_by_disease'].head(20))
display(source_specific['status_by_reason'])
display(source_specific['reports_per_event'].head(20).to_frame('reports'))

In [ ]:
df.sort_values('submissionDate', ascending=False)[
    ['country', 'disease', 'subType', 'eventStartDate', 'submissionDate', 'reason', 'reportStatus', 'eventId', 'reportId']
].head(25)

## Quick Filters

Set any filter to a string value, or leave it as `None` to ignore that filter.

In [ ]:
country_filter = None
disease_filter = None
reason_filter = None
subtype_filter = None

filtered = df.copy()
if country_filter:
    filtered = filtered[filtered['country'].eq(country_filter)]
if disease_filter:
    filtered = filtered[filtered['disease'].eq(disease_filter)]
if reason_filter:
    filtered = filtered[filtered['reason'].eq(reason_filter)]
if subtype_filter:
    filtered = filtered[filtered['subType'].eq(subtype_filter)]

filtered.sort_values('submissionDate', ascending=False).head(50)